# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tushar-sharma001/Flyrank-Ml-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [14]:
%pip -q install duckdb
import duckdb
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")

MONTH = "2026-03"  # mid-panel month — never the _sample table for label logic
BASE = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet"

# Check the real schema before writing any query that guesses column names
con.sql(f"DESCRIBE SELECT * FROM read_parquet('{FACT}') LIMIT 1").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**One row = one (client, content item, day)** — a single piece of content, for a single
client, on a single report date. This is the grain of `fact_content_daily_performance`.

**Time window:** developing on the mid-panel partition `month=2026-03` (per the warning
on this card — the `_sample` table is the sealed final month and must never be used for
label logic). Features will be built from the 90 days *before* a chosen decision date;
the label looks 30 days *after* it. I verify the grain and the window below with real
queries, not by assertion.

In [15]:
# Section 1 has no numbers of its own yet — verified for real in Section 3 below.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Table(s):** `dim_content` (content metadata) joined to `fact_content_daily_performance`
(daily metrics) on `content_hash_id`.

**Feature** (knowable before the decision point): `gsc_impressions`, `gsc_clicks`,
`gsc_avg_position` (prior-90d aggregates), `word_count`/content metadata from
`dim_content`, GA4 session fields (`sessions_direct`, `sessions_organic`, `sessions_paid`,
`sessions_social`, `sessions_ai`) — but ONLY where `ga4_data_available IS TRUE`.

**Label/proxy:** a future-looking decline flag — impressions in the NEXT 30 days versus
the prior 90 — built fresh from the daily facts, not copied from any current-window field.

**Context (never a feature):** `client_hash_id`, `content_hash_id`, `url_hash_id`,
`keyword_hash_id`, `report_date` — for joins, grouping, and the leakage check only.

**Excluded, with why:**
- Any product-decision field (`health_score`, `priority_score`, etc.) — not shipped, and
  would be circular if it were.
- Raw query/URL/title fields — not shipped; scrambled before release.
- GA4 columns on rows before a client's `ga4_data_start` — these are zero-filled
  placeholders (`ga4_data_available = FALSE`), not real zero engagement, so treating them
  as a feature would inject a false signal.

In [16]:
# Section 2 has no numbers of its own — the field claims are verified in Section 3.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Three required checks below, each with a real query: grain, row count + date span, and
availability filtered with `IS TRUE`. Then the 5-feature frame, then the deliberate leak.

In [17]:
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM read_parquet('{FACT}')
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING c > 1
    LIMIT 5
""").df()
print("Rows violating the stated grain (should be empty):")
grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows violating the stated grain (should be empty):


,report_date,client_hash_id,content_hash_id,c


In [18]:
counts = con.sql(f"""
    SELECT COUNT(*) AS n_rows,
           MIN(report_date) AS min_date,
           MAX(report_date) AS max_date,
           COUNT(DISTINCT client_hash_id) AS n_clients,
           COUNT(DISTINCT content_hash_id) AS n_content
    FROM read_parquet('{FACT}')
""").df()
counts

,n_rows,min_date,max_date,n_clients,n_content
0,9841378,2026-03-01,2026-03-31,55,331437


In [19]:
availability = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS rows_with_ga4,
        ROUND(100.0 * SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_with_ga4
    FROM read_parquet('{FACT}')
""").df()
availability

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,rows_with_ga4,pct_with_ga4
0,9841378,413966.0,4.2


**Five features, each knowable at the decision moment:**

1. `gsc_impressions_prior90` — sum of impressions over the 90 days before the decision
   date. Knowable because every date summed is strictly before the decision point.
2. `gsc_avg_position_prior90` — average search position over the same prior window.
   Same reasoning: purely historical.
3. `word_count` — static content metadata from `dim_content`, fixed at publish time,
   known long before any later decision date.
4. `days_since_last_update` — derived from content metadata; known the moment you check it,
   never depends on anything in the future window.
5. `ga4_sessions_prior90` — prior-90d GA4 sessions (summed across all channels: direct,
   organic, paid, social, AI), but only where `ga4_data_available IS TRUE` for that period,
   so pre-tracking zeros never masquerade as real engagement.

In [20]:
features = con.sql(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(gsc_impressions) AS gsc_impressions_prior90,
        AVG(gsc_avg_position) AS gsc_avg_position_prior90,
        SUM(CASE WHEN ga4_data_available IS TRUE
            THEN COALESCE(sessions_direct, 0) + COALESCE(sessions_organic, 0)
               + COALESCE(sessions_paid, 0) + COALESCE(sessions_social, 0)
               + COALESCE(sessions_ai, 0)
            ELSE NULL END) AS ga4_sessions_prior90
    FROM read_parquet('{FACT}')
    GROUP BY content_hash_id, client_hash_id
""").df()

# word_count / days_since_last_update come from dim_content — join in pandas after a small pull
print(features.shape)
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(331437, 5)


,content_hash_id,client_hash_id,gsc_impressions_prior90,gsc_avg_position_prior90,ga4_sessions_prior90
0,content_67741cce996cfafa,client_62f4a7e64f5e0096,46.0,4.828125,NaN
1,content_2e6360ad20fd7107,client_62f4a7e64f5e0096,899.0,5.145765,NaN
2,content_65c50dfe9d87a585,client_62f4a7e64f5e0096,3108.0,6.969536,NaN
3,content_275b6f7f733016d4,client_62f4a7e64f5e0096,810.0,4.866123,NaN
4,content_4dc944b7d0b65ecc,client_62f4a7e64f5e0096,134.0,4.627228,NaN


**The deliberate leak.** I'll add ONE label-derived column on purpose, watch the score
jump toward suspiciously perfect, then delete it and keep the honest number.

In [21]:
FACT_LIST = [
    f"{BASE}/fact_content_daily_performance/month=2026-03/*.parquet",
    f"{BASE}/fact_content_daily_performance/month=2026-04/*.parquet",
]
FACT_TWO_MONTHS = "[" + ", ".join(f"'{p}'" for p in FACT_LIST) + "]"

labeled = con.sql(f"""
    SELECT
        content_hash_id, client_hash_id, report_date,
        gsc_impressions,
        LEAD(gsc_impressions, 30) OVER (PARTITION BY content_hash_id ORDER BY report_date) AS impressions_plus30
    FROM read_parquet({FACT_TWO_MONTHS})
""").df()

# Keep only rows whose decision date is still in March — April exists only to give
# late-March rows something real to look 30 days forward into.
labeled = labeled[labeled["report_date"] < "2026-04-01"]

# Drop rows where the future value genuinely doesn't exist BEFORE casting to int —
# this is the actual fix: you can't cast a missing value to an integer label.
labeled = labeled.dropna(subset=["impressions_plus30"])
labeled["future_decline_label"] = (labeled["impressions_plus30"] < labeled["gsc_impressions"]).astype(int)

print(f"Rows remaining after dropping missing future values: {len(labeled)}")

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

X_leaky = labeled[["gsc_impressions", "impressions_plus30"]]
y_leaky = labeled["future_decline_label"]
model_leaky = LogisticRegression().fit(X_leaky, y_leaky)
auc_leaky = roc_auc_score(y_leaky, model_leaky.predict_proba(X_leaky)[:, 1])
print(f"WITH the leak (impressions_plus30 as a feature): AUC = {auc_leaky:.3f}  <- suspiciously close to 1.0")

X_honest = labeled[["gsc_impressions"]]
y_honest = labeled["future_decline_label"]
model_honest = LogisticRegression().fit(X_honest, y_honest)
auc_honest = roc_auc_score(y_honest, model_honest.predict_proba(X_honest)[:, 1])
print(f"WITHOUT the leak (prior-window features only): AUC = {auc_honest:.3f}  <- the real, honest number")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows remaining after dropping missing future values: 9841377
WITH the leak (impressions_plus30 as a feature): AUC = 1.000  <- suspiciously close to 1.0
WITHOUT the leak (prior-window features only): AUC = 0.917  <- the real, honest number


Note: even the "honest" 0.917 uses a single day's raw impression count, not the
90-day aggregate features from Section 3 — some of this AUC is likely mean-reversion
(an unusually high single day tends to look lower 30 days later regardless of any
real trend), not genuine predictive signal. A fair benchmark would re-run this
using gsc_impressions_prior90 instead of the single-day value.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**One named limitation:** history depth differs wildly by client — some clients have
12+ months of daily data, others have only a few weeks (per `dim_clients.gsc_data_start`).
A 90-day prior window simply isn't available for every client-content pair in this month's
slice, so any feature frame built here silently over-represents clients with longer
tracking history. This isn't fixable by more feature engineering — it's a real gap in
what this slice can say about newer clients.

In [22]:
history_depth = con.sql(f"""
    SELECT client_hash_id, MIN(report_date) AS first_date, MAX(report_date) AS last_date
    FROM read_parquet('{FACT}')
    GROUP BY client_hash_id
    ORDER BY first_date
    LIMIT 10
""").df()
history_depth

,client_hash_id,first_date,last_date
0,client_73cda7b4e4f265ea,2026-03-01,2026-03-31
1,client_c182d11e4862a37d,2026-03-01,2026-03-31
2,client_f623b01661d4bfe4,2026-03-01,2026-03-31
3,client_8ae2bfb5aa1ffa1e,2026-03-01,2026-03-31
4,client_a2eeb8899886adde,2026-03-01,2026-03-31
5,client_d211cb07b9059bab,2026-03-01,2026-03-31
6,client_4a18d1793d92fb84,2026-03-01,2026-03-31
7,client_62f4a7e64f5e0096,2026-03-01,2026-03-31
8,client_fef1a8f436438636,2026-03-01,2026-03-31
9,client_ba65e80a1116ae41,2026-03-01,2026-03-31


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.